# 哺乳類の系統樹を作ろう
## ミトコンドリア DNA(cytb 遺伝子)から「海に戻った哺乳類」の進化を読み解く

このノートブックでは、**37種類の哺乳類** のミトコンドリア DNA から系統樹を作り、
「**陸上から海へ何度も独立に進出した**」という哺乳類進化のドラマを観察します。

### このデータセットの特徴

鳥類のデータセットと違い、これは **海生哺乳類とその陸上の親戚** にフォーカスした厳選セットです。

| 系統 | 種数 | 例 |
| --- | --- | --- |
| **Mysticeti**(ヒゲクジラ亜目) | 7 | シロナガス、ザトウクジラ、コクなど |
| **Odontoceti**(ハクジラ亜目) | 15 | マッコウ、シャチ、イルカ、イッカク、カワイルカなど |
| **Pinnipedia**(鰭脚類) | 6 | アザラシ、アシカ、トド、セイウチ |
| **Sirenia**(海牛目) | 2 | マナティー、ジュゴン |
| **Mustelidae**(イタチ科) | 1 | ラッコ |
| **Ursidae**(クマ科) | 1 | ホッキョクグマ |
| **Terrestrial-sister**(陸上の姉妹群) | 5 | カバ、ヒグマ、カワウソ、ゾウ、ハイラックス |

### 解析の 2 段構え

1. **入門編** — Python だけで完結する単純なアプローチ(切り詰めアラインメント + 近隣結合法)
2. **本格編** — 研究現場の標準ツール **MAFFT** + **IQ-TREE** で **最尤法** + ブートストラップ解析

### 注意:外群がない

鳥類データセットでは「ミシシッピワニ」を外群として使えましたが、今回はすべて哺乳類です。
そこで **中点ルーティング (midpoint rooting)** ——「最も遠いペアの中点」を根とする方法 ——で系統樹を根づけします。

> 大学1年生の一般教養課程を想定し、専門用語は最小限にしています。
> ライセンス:配列データは NCBI のパブリックデータ、シルエット画像は PhyloPic(CC ライセンス、`mammals/phylopic_mammals/phylopic_attribution.tsv` 参照)。

## 0. 必要ライブラリのインストール(初回のみ)

Python パッケージ:
- `biopython` : 配列解析と系統樹計算
- `matplotlib-fontja` : 日本語フォント

**MAFFT と IQ-TREE は別途インストールが必要です(後のセクション 12 で扱います)。**

In [ ]:
%pip install biopython pandas numpy matplotlib seaborn matplotlib-fontja

## 1. ライブラリの読み込みと日本語フォントの設定

**順序が重要**:seaborn のスタイル設定を先に行い、その後に `matplotlib_fontja` を読み込みます。

In [ ]:
import sys
import shutil
import subprocess
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
import seaborn as sns

from Bio import SeqIO, AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

# 日本語フォントの設定(順序が重要)
sns.set_style("whitegrid")
try:
    import matplotlib_fontja  # noqa: F401
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError:
    print("matplotlib-fontja が見つかりません。上のセルでインストールしてください。")
plt.rcParams["axes.unicode_minus"] = False

# データファイルへのパス(このノートブックは ml/ から実行する想定)
MAMMALS_DIR = Path("../mammals")
FASTA_PATH = MAMMALS_DIR / "mammals_cytb.fasta"
METADATA_PATH = MAMMALS_DIR / "mammals_cytb_metadata.tsv"
PHYLOPIC_DIR = MAMMALS_DIR / "phylopic_mammals" / "png"

# MAFFT / IQ-TREE 用の作業ディレクトリ
WORK_DIR = Path("./phylo_work_mammals")
WORK_DIR.mkdir(exist_ok=True)

for p in [FASTA_PATH, METADATA_PATH, PHYLOPIC_DIR]:
    assert p.exists(), f"見つかりません: {p}"
print(f"データ準備 OK: {MAMMALS_DIR.resolve()}")
print(f"作業ディレクトリ: {WORK_DIR.resolve()}")

## 2. FASTA ファイルの読み込み

**FASTA** は配列データを表す基本的なテキスト形式です。

```
>Balaena_mysticetus|NC_005268.1|CYTB    ← ヘッダー
ATGACCAACATCCGAAAAACACAC...              ← 配列本体
```

ヘッダーは `>属_種|NCBI アクセッション番号|遺伝子名` という形式です。

In [ ]:
records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print(f"読み込んだ配列数: {len(records)}")
print(f"最初のヘッダー : {records[0].id}")
print(f"最初の60塩基   : {str(records[0].seq)[:60]} ...")

# ヘッダーから「種名(属_種)」だけを取り出して ID を整理
for rec in records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""

print(f"\n整理後の ID 一覧(先頭5件): {[r.id for r in records[:5]]}")

## 3. メタデータ(和名・分類群)の読み込み

各種の和名・英名・分類群が `mammals_cytb_metadata.tsv` にまとめられています。

**注目ポイント**:

- 同じ「**Cetacea(クジラ目)**」でも、**Mysticeti**(ヒゲクジラ)と **Odontoceti**(ハクジラ)に分かれる
- 「**Carnivora(食肉目)**」の中に、海(Pinnipedia, ラッコ)・氷上(ホッキョクグマ)・陸(ヒグマ、カワウソ)が混在
- 「**Terrestrial-sister**」は、海生グループの**陸上の最近縁種**として明示的に選ばれている(例:カバはクジラの姉妹群)

In [ ]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
metadata["id"] = metadata["scientific_name"].str.replace(" ", "_")

print(f"メタデータ行数: {len(metadata)}")
print("\n各系統の種数:")
print(metadata["clade"].value_counts())
print("\n各目 (order) の種数:")
print(metadata["order"].value_counts())
metadata[["scientific_name", "common_name_ja", "order", "clade"]].head(10)

## 4. 配列の長さと組成を確認

- **配列長** がそろっているか? → そろっていなければ後でアラインメントが必要
- **GC 含量** (G + C の割合) → 系統で傾向があるか?

In [ ]:
def gc_content(seq):
    s = str(seq).upper()
    return (s.count("G") + s.count("C")) / len(s) * 100

seq_stats = pd.DataFrame({
    "id": [r.id for r in records],
    "length": [len(r.seq) for r in records],
    "gc_pct": [gc_content(r.seq) for r in records],
}).merge(metadata[["id", "common_name_ja", "clade"]], on="id", how="left")

print("配列長の統計:")
print(seq_stats["length"].describe().round(1))
print(f"\n配列長の種類: {sorted(seq_stats['length'].unique())}")
seq_stats.head()

In [ ]:
# 系統別の GC 含量(箱ひげ図 + 個々の点)
fig, ax = plt.subplots(figsize=(11, 4))
order = seq_stats.groupby("clade")["gc_pct"].median().sort_values().index
sns.boxplot(data=seq_stats, x="clade", y="gc_pct", order=order,
            color="lightsteelblue", ax=ax)
sns.stripplot(data=seq_stats, x="clade", y="gc_pct", order=order,
              color="darkblue", alpha=0.6, size=4, ax=ax)
ax.set_xlabel("系統 (clade)")
ax.set_ylabel("GC 含量 (%)")
ax.set_title("cytb 遺伝子の GC 含量(系統別)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 5. 入門編 — シンプルなアラインメント

DNA から系統樹を作るには、**「同じ位置に同じ進化的起源の塩基を縦に揃える」** 作業 = **マルチプルアラインメント (MSA)** が必要です。

今回のデータには都合のいい特徴があります。

- すべての配列が **ATG**(開始コドン)から始まっている
- 長さは 1137 〜 1140 塩基とほぼ同じ(脊椎動物の cytb は長さがよく保存されている)

入門編では、簡単のため **全配列の先頭から共通の長さだけを切り出して使う** ことにします。
本格的なアラインメントは後のセクション 12 で MAFFT を使って行います。

In [ ]:
min_len = min(len(r.seq) for r in records)
print(f"全配列を {min_len} 塩基に切り詰めます")

for rec in records:
    rec.seq = rec.seq[:min_len]

alignment = MultipleSeqAlignment(records)
print(f"\nアラインメント:")
print(f"  配列数 : {len(alignment)}")
print(f"  カラム数: {alignment.get_alignment_length()}")

## 6. 遺伝的距離の計算

**同一性距離 (identity distance)** で種同士の遺伝的な「離れぐあい」を測ります。

$$d_{\mathrm{identity}}(A, B) = \frac{\text{異なる塩基の数}}{\text{全塩基数}}$$

In [ ]:
calculator = DistanceCalculator("identity")
dm = calculator.get_distance(alignment)

print(f"距離行列のサイズ: {len(dm.names)} × {len(dm.names)}")
print(f"\n注目の比較:")
print(f"  シロナガス vs ザトウ(どちらもヒゲクジラ)")
print(f"    d = {dm['Balaenoptera_musculus', 'Megaptera_novaeangliae']:.4f}")
print(f"  シロナガス vs マッコウ(ヒゲ vs ハクジラ)")
print(f"    d = {dm['Balaenoptera_musculus', 'Physeter_macrocephalus']:.4f}")
print(f"  シロナガス vs カバ(クジラとカバは姉妹群)")
print(f"    d = {dm['Balaenoptera_musculus', 'Hippopotamus_amphibius']:.4f}")
print(f"  ホッキョクグマ vs ヒグマ(近縁種同士)")
print(f"    d = {dm['Ursus_maritimus', 'Ursus_arctos']:.4f}")
print(f"  マナティー vs アフリカゾウ(マナティーとゾウは同じアフリカ獣類)")
print(f"    d = {dm['Trichechus_manatus', 'Loxodonta_africana']:.4f}")

## 7. 距離行列をヒートマップで可視化

行と列を **系統(clade)順** に並べると、近縁な種同士が「ブロック」を成して見えるはずです。

In [ ]:
names = list(dm.names)
n = len(names)
mat = np.array([[dm[a, b] for b in names] for a in names])
dist_df = pd.DataFrame(mat, index=names, columns=names)

# clade 順に並べ替え
clade_order_list = [
    "Mysticeti", "Odontoceti",          # クジラ
    "Pinnipedia", "Mustelidae", "Ursidae",  # 食肉目の海生・半海生
    "Sirenia",                          # 海牛目
    "Terrestrial-sister",               # 陸上姉妹群
]
id_to_clade = metadata.set_index("id")["clade"].to_dict()
ordered = sorted(names, key=lambda x: (clade_order_list.index(id_to_clade.get(x, "Terrestrial-sister")), x))
dist_df = dist_df.loc[ordered, ordered]

ja_map = metadata.set_index("id")["common_name_ja"].to_dict()
labels_ja = [ja_map.get(x, x) for x in dist_df.index]

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(dist_df, cmap="viridis",
            xticklabels=labels_ja, yticklabels=labels_ja,
            cbar_kws={"label": "同一性距離"}, ax=ax, square=True)
ax.set_title("哺乳類37種間の遺伝的距離(系統順に並べ替え)")
plt.tight_layout()
plt.show()

**観察ポイント**

- クジラ目(Mysticeti + Odontoceti)はひとつの大きな「濃いブロック」を作っている
- 鰭脚類(Pinnipedia)も互いに近い
- でも **クジラ全体 vs 鰭脚類** はかなり離れている(別系統の海生哺乳類)
- ラッコ(Mustelidae)は **カワウソ**(Terrestrial-sister)と近い色をしているはず

## 8. 近隣結合法 (NJ法) で系統樹を作る

**Neighbor Joining (NJ) 法** は距離行列から系統樹を作る代表的な方法です。
1987年に斎藤・根井(日本人研究者)により発表され、現在も広く使われています。

**アルゴリズムのアイデア(直感)**

1. 距離行列をもとに「**最も近い 2 種**」を選んで枝でつなぐ
2. その 2 種を 1 つのグループにまとめ、他の種からの距離を更新
3. 1〜2 を繰り返し、すべての種が 1 つの木に統合されるまで続ける

### 外群がないので「中点ルーティング」で根づけ

今回のデータには外群(明らかに他から離れている種)がありません。
そこで **中点ルーティング (midpoint rooting)** ——「最も遠い 2 種を結ぶパスの中点」を根とする方法 ——を使います。
進化速度がだいたい一定という仮定が必要ですが、外群が手元にない場合の標準的な方法です。

In [ ]:
constructor = DistanceTreeConstructor()
nj_tree = constructor.nj(dm)

# 中点ルーティング
nj_tree.root_at_midpoint()

# 内部ノードのデフォルト名(Inner1, Inner2, ...)を消す
for clade in nj_tree.get_nonterminals():
    clade.name = None

print(f"NJ 系統樹を作成しました")
print(f"  末端ノード(種)の数: {nj_tree.count_terminals()}")

## 9. 系統樹の描画(まずはシンプルに)

Biopython の `Phylo.draw` を使い、和名ラベル + 系統別の色 で描画します。

In [ ]:
# 海生 = 寒色系、陸生 = 暖色系で配色
clade_to_color = {
    "Mysticeti":          "#1f4e79",  # 濃紺 — ヒゲクジラ
    "Odontoceti":         "#2e86ab",  # 青  — ハクジラ
    "Pinnipedia":         "#1abc9c",  # 緑青 — 鰭脚類
    "Sirenia":            "#16a085",  # 深緑 — 海牛
    "Mustelidae":         "#d35400",  # 茶  — ラッコ
    "Ursidae":            "#7d6608",  # 茶金 — ホッキョクグマ
    "Terrestrial-sister": "#7f8c8d",  # 灰 — 陸上姉妹群
}
id_to_color = {row["id"]: clade_to_color.get(row["clade"], "#000000")
               for _, row in metadata.iterrows()}
ja_to_color = {row["common_name_ja"]: clade_to_color.get(row["clade"], "#000000")
               for _, row in metadata.iterrows()}

def label_func(clade):
    if clade.is_terminal():
        return ja_map.get(clade.name, clade.name)
    return ""

fig, ax = plt.subplots(figsize=(11, 15))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=ax,
               label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
ax.set_title("哺乳類 cytb の NJ 系統樹(中点ルーティング)")
legend = [Patch(facecolor=c, label=cl) for cl, c in clade_to_color.items()]
ax.legend(handles=legend, loc="lower right", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()

## 10. シルエット付きの系統樹

PhyloPic のシルエット画像を末端に並べ、より直感的に読める樹を作ります。
シルエットは系統色で塗り分けます。

> PhyloPic ([phylopic.org](https://www.phylopic.org/)) は生物のシルエットを Creative Commons ライセンスで配布しているプロジェクトです。
> 帰属情報は `mammals/phylopic_mammals/phylopic_attribution.tsv` を参照。

In [ ]:
# 系統樹を描画するための座標計算
def get_x_positions(tree):
    depths = tree.depths()
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths

def get_y_positions(tree):
    max_h = tree.count_terminals()
    heights = {tip: max_h - i for i, tip in enumerate(tree.get_terminals())}
    def calc(clade):
        for sub in clade:
            if sub not in heights:
                calc(sub)
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0
    if tree.root.clades:
        calc(tree.root)
    return heights

def colorize_silhouette(img, color):
    """黒シルエット PNG を指定色に塗り替える(アルファチャンネルは保持)"""
    if img.ndim != 3 or img.shape[2] != 4:
        return img
    rgb = to_rgb(color)
    colored = np.zeros_like(img)
    colored[..., 0] = rgb[0]
    colored[..., 1] = rgb[1]
    colored[..., 2] = rgb[2]
    colored[..., 3] = img[..., 3]
    return colored

def draw_tree_with_silhouettes(tree, title, xlabel="遺伝的距離", show_bootstrap=False):
    """系統樹を PhyloPic シルエット付きで描画する共通関数"""
    xpos = get_x_positions(tree)
    ypos = get_y_positions(tree)
    x_max = max(xpos.values())

    fig, ax = plt.subplots(figsize=(14, 18))

    def draw_branch(clade, x_start):
        x_here, y_here = xpos[clade], ypos[clade]
        col = id_to_color.get(clade.name, "#888") if clade.is_terminal() else "#888"
        ax.plot([x_start, x_here], [y_here, y_here], color=col, linewidth=1.6)
        if clade.clades:
            ys = [ypos[c] for c in clade.clades]
            ax.plot([x_here, x_here], [min(ys), max(ys)], color="#888", linewidth=1.6)
            for sub in clade.clades:
                draw_branch(sub, x_here)

    draw_branch(tree.root, 0)

    # 末端にシルエットと和名を配置
    for tip in tree.get_terminals():
        y, x = ypos[tip], xpos[tip]
        species = tip.name
        color = id_to_color.get(species, "#000")
        img_path = PHYLOPIC_DIR / f"{species}.png"
        if img_path.exists():
            img = colorize_silhouette(mpimg.imread(img_path), color)
            ab = AnnotationBbox(OffsetImage(img, zoom=0.04),
                                (x + x_max * 0.10, y),
                                xycoords="data", frameon=False)
            ax.add_artist(ab)
        ja = ja_map.get(species, species)
        ax.text(x + x_max * 0.15, y, ja, va="center", fontsize=10, color=color)

    # 内部ノードにブートストラップ値を表示(ある場合)
    if show_bootstrap:
        for clade in tree.get_nonterminals():
            if clade.confidence is None or clade == tree.root:
                continue
            bs = clade.confidence
            bc = "#1b9e77" if bs >= 95 else ("#d95f02" if bs >= 70 else "#b22222")
            ax.text(xpos[clade], ypos[clade], f"{bs:.0f}",
                    fontsize=8, ha="right", va="center", color=bc,
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                              edgecolor="none", alpha=0.8))

    ax.set_xlim(-x_max * 0.02, x_max * 1.35)
    ax.set_ylim(0, tree.count_terminals() + 1)
    ax.set_yticks([])
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=13)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

    handles = [Patch(facecolor=c, label=cl) for cl, c in clade_to_color.items()]
    if show_bootstrap:
        handles += [
            Patch(facecolor="#1b9e77", label="BS ≥ 95"),
            Patch(facecolor="#d95f02", label="BS 70–94"),
            Patch(facecolor="#b22222", label="BS < 70"),
        ]
    ax.legend(handles=handles, loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()

draw_tree_with_silhouettes(nj_tree, "哺乳類 cytb の NJ 系統樹(PhyloPic シルエット付き)")

## 11. 別の方法(UPGMA)と比較してみる

**UPGMA** は NJ よりさらにシンプルな方法で、「距離が最も近いペアを順次まとめていく」だけです。
ただし **すべての系統で進化速度が一定** という強い仮定を置くため、NJ よりも精度が落ちることが知られています。

In [ ]:
upgma_tree = constructor.upgma(dm)
for clade in upgma_tree.get_nonterminals():
    clade.name = None

fig, axes = plt.subplots(1, 2, figsize=(20, 16))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=axes[0],
               label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
    axes[0].set_title("NJ (Neighbor Joining)")
    Phylo.draw(upgma_tree, label_func=label_func, do_show=False, axes=axes[1],
               label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
    axes[1].set_title("UPGMA")
plt.tight_layout()
plt.show()

## 12. 本格編 — MAFFT でアラインメントし、IQ-TREE で最尤系統樹を推定

ここからは、研究現場で実際に使われている **業界標準ツール** で同じデータを解析し直します。

| ツール | 役割 | 入門編との違い |
| --- | --- | --- |
| **MAFFT** | マルチプルアラインメント | 配列を端で切るだけでなく、必要に応じて **ギャップ(挿入・欠失)** を入れて精密に揃える |
| **IQ-TREE** | 最尤法による系統樹推定 | 距離法ではなく、塩基置換モデルに基づく **確率的推論**。**ブートストラップ** で各枝の信頼度も算出 |

これらは Python ライブラリではなく **独立した実行ファイル(コマンドラインツール)** です。

### 12.1 MAFFT と IQ-TREE のインストール

環境に応じて以下のいずれかを実行してください(初回のみ)。

```bash
# Google Colab・Ubuntu/Debian
!apt-get install -y mafft iqtree

# conda 環境(より新しい IQ-TREE が手に入る)
conda install -c bioconda mafft iqtree

# macOS (Homebrew)
brew install mafft iqtree
```

> Note: 環境によって IQ-TREE のコマンド名は `iqtree`, `iqtree2`, `iqtree3` のいずれかです。
> 以下のコードでは自動検出し、バージョンに応じてオプションも切り替えます。

In [ ]:
# Colab で直接インストールしたいときは下のコメントを外す
# !apt-get install -y mafft iqtree

def find_command(candidates):
    for cmd in candidates:
        path = shutil.which(cmd)
        if path:
            return cmd, path
    return None, None

mafft_cmd, mafft_path = find_command(["mafft"])
iqtree_cmd, iqtree_path = find_command(["iqtree3", "iqtree2", "iqtree"])

print(f"MAFFT  : {mafft_path or '見つかりません'}")
print(f"IQ-TREE: {iqtree_path or '見つかりません'} (コマンド名: {iqtree_cmd})")

if not (mafft_cmd and iqtree_cmd):
    print("\n⚠ どちらかが見つからない場合は、上のセルに従ってインストールしてください。")
    print("  以下の MAFFT / IQ-TREE のセクションはスキップされ、NJ 系統樹のみ得られます。")

### 12.2 MAFFT で配列をアラインメント

**MAFFT** は速くて精度の高いマルチプルアラインメントツール(京都大学・加藤らが開発)です。
`--auto` オプションで最適なアルゴリズムを自動選択します。

In [ ]:
# 整理済み ID で FASTA を再生成(MAFFT の入力に使う)
stripped_fasta = WORK_DIR / "mammals_cytb_stripped.fasta"
clean_records = list(SeqIO.parse(FASTA_PATH, "fasta"))
for rec in clean_records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""
SeqIO.write(clean_records, stripped_fasta, "fasta")
print(f"ID 整理済み FASTA: {stripped_fasta}")

aligned_path = WORK_DIR / "mammals_cytb_aligned.fasta"

if mafft_cmd:
    print("\nMAFFT を実行中(数秒で終わります)...")
    with open(aligned_path, "w") as f:
        subprocess.run(
            [mafft_cmd, "--auto", "--quiet", str(stripped_fasta)],
            stdout=f, stderr=subprocess.PIPE, check=True, text=True,
        )
    print(f"完了: {aligned_path}")
else:
    print("MAFFT がインストールされていません。スキップします。")

### 12.3 アラインメント結果を確認

MAFFT は挿入・欠失(ギャップ `-`)を入れて配列をきれいに揃えます。

In [ ]:
if aligned_path.exists():
    mafft_alignment = AlignIO.read(aligned_path, "fasta")
    print(f"配列数              : {len(mafft_alignment)}")
    print(f"カラム数(MAFFT後)  : {mafft_alignment.get_alignment_length()}")
    print(f"カラム数(切り詰めのみ): {alignment.get_alignment_length()}")
    diff = mafft_alignment.get_alignment_length() - alignment.get_alignment_length()
    print(f"  → MAFFT は {diff:+d} カラム分のギャップを追加")
    
    print("\n最初の60カラム × 5配列:")
    for rec in mafft_alignment[:5]:
        print(f"  {rec.id:30s} {str(rec.seq)[:60]}")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")
    mafft_alignment = None

### 12.4 保存性プロファイル(どの位置がどれだけ変わったか)

各カラムで「最も多い塩基がどれだけ多数派か」 = **保存性スコア** を計算します。

- 保存率の **高い** 領域 = タンパク質の機能的に重要な部位
- 保存率の **低い** 領域 = 中立進化に近く、種ごとに多様な部位

In [ ]:
if mafft_alignment is not None:
    aln_array = np.array([list(str(rec.seq).upper()) for rec in mafft_alignment])
    n_seqs, n_cols = aln_array.shape
    
    conservation = np.zeros(n_cols)
    for j in range(n_cols):
        col = aln_array[:, j]
        col_no_gap = col[col != "-"]
        if len(col_no_gap) > 0:
            _, counts = np.unique(col_no_gap, return_counts=True)
            conservation[j] = counts.max() / n_seqs
    
    window = 30
    smoothed = np.convolve(conservation, np.ones(window)/window, mode="valid")
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(range(len(smoothed)), smoothed, color="steelblue", linewidth=1.5)
    ax.axhline(0.9, color="red", linestyle="--", alpha=0.5, label="保存率 90%")
    ax.axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="保存率 50%")
    ax.set_xlabel(f"アラインメント上のカラム位置({window}カラム移動平均)")
    ax.set_ylabel("保存性スコア")
    ax.set_title(f"cytb 遺伝子の保存性プロファイル({n_seqs}種、{n_cols}カラム)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n保存率 ≥ 90% のカラム: {(conservation >= 0.9).sum()} / {n_cols} "
          f"({100*(conservation >= 0.9).mean():.1f}%)")
    print(f"保存率 < 50% のカラム: {(conservation < 0.5).sum()} / {n_cols} "
          f"({100*(conservation < 0.5).mean():.1f}%)")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")

### 12.5 最尤法と IQ-TREE

**最尤法 (Maximum Likelihood, ML)** は、データを生み出す確率モデルに基づいて、
「観察された配列を最もよく説明する系統樹」を統計的に探す方法です。

| | 距離法 (NJ) | 最尤法 (IQ-TREE) |
| --- | --- | --- |
| 入力 | 距離行列(配列の情報を要約) | アラインメント(全情報) |
| 塩基置換モデル | なし、または最初に固定 | **データから最適なモデルを自動選択** |
| 計算量 | 軽い(数秒) | 重い(37種で数十秒〜2 分) |
| 信頼度 | 通常はなし | **ブートストラップ**で各枝の支持率を算出 |

**IQ-TREE のオプション**

- `-s` : 入力アラインメント
- `-m MFP` : ModelFinder Plus(最適な塩基置換モデルを自動選択)
- `-B 1000` (v2) / `-bb 1000` (v1) : UltraFast Bootstrap を 1000 回
- `-T AUTO` (v2) / `-nt AUTO` (v1) : スレッド数の自動設定
- `-redo` : 既存の出力を上書き

**ブートストラップ** とは、アラインメントのカラムをランダムに重複抽出し直して系統樹を 1000 回作り直し、
各枝が何 % の試行で再現されたかを「**支持率**」として返す仕組みです。

In [ ]:
ml_treefile = Path(str(aligned_path) + ".treefile")

if iqtree_cmd and aligned_path.exists():
    # IQ-TREE のバージョンを検出してオプションを切り替える
    help_out = subprocess.run([iqtree_cmd, "--help"], capture_output=True, text=True)
    help_text = help_out.stdout + help_out.stderr
    is_modern = ("-B NUM" in help_text) or ("--prefix" in help_text)
    
    bootstrap_args = ["-B", "1000"] if is_modern else ["-bb", "1000"]
    threads_args = ["-T", "AUTO"] if is_modern else ["-nt", "AUTO"]
    print(f"IQ-TREE バージョン: {'v2/v3 系' if is_modern else 'v1 系'}")
    
    print(f"IQ-TREE を実行中(30秒〜2分)...")
    proc = subprocess.run(
        [iqtree_cmd, "-s", str(aligned_path), "-m", "MFP",
         *bootstrap_args, *threads_args, "-redo"],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        print("IQ-TREE 完了")
        print("\n生成されたファイル:")
        for f in sorted(WORK_DIR.glob("mammals_cytb_aligned.fasta.*")):
            print(f"  {f.name}")
    else:
        print(f"IQ-TREE がエラーで終了しました:\n{proc.stderr[-500:]}")
else:
    print("IQ-TREE またはアラインメントが利用できないため、スキップします。")

### 12.6 採用されたモデルを確認

ModelFinder は何十種類もの塩基置換モデルから AIC/BIC で最良のものを選びます。

In [ ]:
report_file = Path(str(aligned_path) + ".iqtree")
if report_file.exists():
    text = report_file.read_text()
    for keyword in ["Best-fit model", "Model of substitution"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
    for keyword in ["Log-likelihood of the tree", "Unconstrained log-likelihood",
                    "Akaike information criterion (AIC) score",
                    "Bayesian information criterion (BIC) score"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
else:
    print("IQ-TREE レポートがないため、このセルはスキップします。")

### 12.7 ML 系統樹の読み込み

IQ-TREE の出力 `.treefile`(Newick 形式)を Biopython で読み込み、**中点ルーティング** で根づけします。
Newick の内部ノードラベルには **ブートストラップ支持率 (0〜100)** が格納されています。

In [ ]:
if ml_treefile.exists():
    ml_tree = Phylo.read(ml_treefile, "newick")
    ml_tree.root_at_midpoint()
    
    # Newick の内部ノードラベル(=ブートストラップ値の文字列)を confidence に移す
    for clade in ml_tree.get_nonterminals():
        if clade.name:
            try:
                clade.confidence = float(clade.name)
            except ValueError:
                pass
        clade.name = None
    
    bs_values = [c.confidence for c in ml_tree.get_nonterminals() if c.confidence is not None]
    print(f"ML 系統樹を読み込みました")
    print(f"  末端: {ml_tree.count_terminals()}")
    if bs_values:
        print(f"  ブートストラップ支持率: 平均 {np.mean(bs_values):.1f}, "
              f"範囲 [{min(bs_values):.0f}, {max(bs_values):.0f}]")
        print(f"  ≥ 95(強い支持)の枝: {sum(b >= 95 for b in bs_values)} / {len(bs_values)}")
else:
    print("ML 系統樹ファイルがないため、このセルはスキップします。")
    ml_tree = None

### 12.8 ML 系統樹をブートストラップ支持率付きで描画

各内部ノードにブートストラップ値を表示します。

- 🟢 緑 (≥ 95): 強い支持(その枝はほぼ確実)
- 🟠 橙 (70–94): 中程度の支持
- 🔴 赤 (< 70): 弱い支持(その枝は信頼できない)

In [ ]:
if ml_tree is not None:
    draw_tree_with_silhouettes(
        ml_tree,
        title="哺乳類 cytb の ML 系統樹(IQ-TREE + UltraFast Bootstrap × 1000)",
        xlabel="塩基置換数 / サイト",
        show_bootstrap=True,
    )
else:
    print("ML 系統樹がないため、このセルはスキップします。")

### 12.9 NJ と ML を並べて比較

In [ ]:
if ml_tree is not None:
    fig, axes = plt.subplots(1, 2, figsize=(20, 16))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=axes[0],
                   label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
        axes[0].set_title("NJ(切り詰めアラインメント + 同一性距離)")
        Phylo.draw(ml_tree, label_func=label_func, do_show=False, axes=axes[1],
                   label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
        axes[1].set_title("ML(MAFFT + IQ-TREE 最尤法 + UFBoot 1000)")
    plt.tight_layout()
    plt.show()
else:
    print("ML 系統樹がないため比較できません。")

## 13. 考察 — 哺乳類はいかにして海に戻ったか

ML 系統樹を眺めながら、哺乳類進化の重要なテーマについて考えてみましょう。

### Q1. 海への独立進出は何回起きた?

「水中生活への適応(流線型の体・ヒレ・水中呼吸)」は、系統樹を見ると **少なくとも 3 回独立に進化** したことが分かるはずです。

| 海生グループ | 進出時期 | 陸上の最近縁種 |
| --- | --- | --- |
| **Cetacea**(クジラ目) | 約 5000 万年前 | **カバ**(Hippopotamus) |
| **Sirenia**(海牛目) | 約 5000 万年前 | **ゾウ・ハイラックス**(アフリカ獣類) |
| **Pinnipedia**(鰭脚類) | 約 3000 万年前 | **クマ・カワウソ**などの食肉目 |

これらが樹上で **「別々の枝」** として現れ、それぞれの近縁な陸生種と一緒のグループに入っているはずです。
これが **収斂進化 (convergent evolution)** の最も有名な例の一つです。

### Q2. クジラの最近縁は誰?——分子系統学の大発見

形態だけで分類していた時代、クジラは「特殊な独立した目」と考えられていました。
しかし 1990 年代以降の分子系統解析で、**クジラの最も近い親戚はカバ** であることが明らかになりました。

→ 今回の樹で、**カバ** はどのグループの近くに来ていますか?
→ この枝のブートストラップ支持率はどうですか?

現在では「クジラ + 偶蹄目」を合わせて **Cetartiodactyla(鯨偶蹄目)** と呼びます。

### Q3. ヒゲクジラとハクジラはどこで分かれた?

クジラ目は約 3500 万年前に **Mysticeti(ヒゲクジラ:濾過摂食)** と **Odontoceti(ハクジラ:エコーロケーション)** に分かれました。

→ 樹の中で 2 系統が明確に分離していますか?
→ Odontoceti の中に **カワイルカ**(Platanista, Inia, Lipotes)はどう入っていますか?
  実はカワイルカは **海から川に何度も独立に進出** したと考えられており、単系統群ではないことが知られています。

### Q4. アシカ・アザラシ・セイウチの関係

鰭脚類(Pinnipedia)は3つの科に分かれます。

- **Otariidae**(アシカ科): カリフォルニアアシカ、トド
- **Phocidae**(アザラシ科): ゴマフアザラシ、ハイイロアザラシ、ミナミゾウアザラシ
- **Odobenidae**(セイウチ科): セイウチ

→ 樹の中で 3 グループに分かれていますか?セイウチはアシカ系・アザラシ系どちらに近いですか?

### Q5. 北極の二人 — ホッキョクグマとホッキョククジラ

**ホッキョクグマ(Ursus maritimus)** と **ヒグマ(Ursus arctos)** は同じ属で、非常に最近(約 50 万年前)分かれた近縁種です。

→ 樹の中で 2 種は隣合っていますか?枝の長さはとても短いはずです。

### Q6. アフリカ獣類(Paenungulata)の絆

**マナティー・ジュゴン**(Sirenia)、**ゾウ**、**ハイラックス**——
見た目はまったく違うこの 3 グループは、すべて **アフリカで進化した古い系統**(Paenungulata)で互いに最も近縁です。

→ 樹の中で 5 種が 1 つのまとまりを作っていますか?この発見もまた、分子系統学が形態学を覆した例です。

## 14. この解析の限界と注意点

MAFFT + IQ-TREE は研究レベルの解析パイプラインですが、それでも完全ではありません。

1. **1 つの遺伝子しか使っていない**
    - cytb はミトコンドリア DNA の一部にすぎず、種の真の系統("species tree")を完全には反映しません
    - 本格的な系統研究では数百〜数千の遺伝子を組み合わせます(**系統ゲノミクス**)

2. **外群がない**
    - 中点ルーティングは「すべての枝で進化速度が大体同じ」という仮定を置く
    - 例えば「ハツカネズミ」「ヒト」など他の哺乳類目をデータに加えれば、もっと信頼できる根づけが可能

3. **ミトコンドリア DNA の偏り**
    - ミトコンドリアは **母親からのみ遺伝**(雑種・交雑が見えない)
    - ホッキョクグマとヒグマは実際には過去に交雑したことが核 DNA から分かっており、mtDNA だけでは見えない歴史がある

4. **長枝誘引 (Long Branch Attraction, LBA)**
    - 急速に進化した(=枝が長い)種同士が、本当は関係なくても「近い」と誤って結ばれることがある
    - とくに進化速度の早い分類群が混ざる場合に注意が必要

## 15. 発展課題

1. **鳥類でやってみる**
    - 同じディレクトリにある `birds_phylogeny_tutorial.ipynb` を参照。鳥類データには **明確な外群(ミシシッピワニ)** があり、根づけの仕方が違うことを比較しよう。

2. **外群を追加して再解析**
    - ハツカネズミ(*Mus musculus*)やヒト(*Homo sapiens*)などの cytb 配列を NCBI から取ってきて、データセットに加えてみよう。
    - 中点ルーティングなしで、明確な外群でルートできるはず。

3. **「クジラ vs カバ」を疑え**
    - 樹を見て「カバ」が本当にクジラ目の隣にいるか、その枝のブートストラップ支持率を確認しよう。
    - 一遺伝子では支持率が低いことが多い。複数遺伝子を使った研究論文(例: Nikaido et al. 1999)と結果を比較してみよう。

4. **時間軸つきの樹を作る**
    - 化石記録から「ヒゲクジラとハクジラの分岐は約 3500 万年前」と分かっています。
    - IQ-TREE の `--date` オプションや `LSD` ソフトで分子時計分析をしてみよう。

5. **異なるブートストラップ法**
    - `-B 1000` (UltraFast Bootstrap, UFBoot) と `-b 100` (Standard Bootstrap) で結果がどう変わるか比較。
    - UFBoot は速いが楽観的、Standard は遅いが保守的。

6. **公開データと比較**
    - [TimeTree](http://www.timetree.org/) や [OneZoom](http://www.onezoom.org/) で本物の哺乳類系統樹を見てみよう。
    - とくに **アフリカ獣類** (Afrotheria) の関係は、近年の系統ゲノミクスでさらに細かく解明されている。

7. **形質マッピング**
    - 樹の各枝に「水生 / 陸生」「肉食 / 草食」などの形質を色塗りすると、形質が何回独立に進化したかが視覚的に分かる。
    - `ete3` などのライブラリで可能。